In [ ]:
from google.cloud import bigquery
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
# Add the name of the project in google console
client = bigquery.Client(project="citric-trees-489611-k5")

# Data Preparation - Patient Table

In [ ]:
# Load the data
patient_query = """
SELECT  pat.subject_id,
        pat.anchor_age

FROM    physionet-data.mimiciv_3_1_hosp.patients pat
"""
# patient_query_job = client.query(patient_query)
# patient_df = patient_query_job.result().to_dataframe()
df_age = client.query(patient_query).to_dataframe()

print(df_age.info())
print(df_age.describe())
print("Nulls:\n", df_age.isnull().sum())

In [ ]:
# Handle the 91+ anonymization artifact
## MIMIC-IV shifts ages >89 to 91 to protect patient identity.
## Two approaches:
### 1) Treat 91 as a sentinel ">=90" value (keep it, acknowledge in docs)
### 2) Clip everything at 89 to remove ambiguity
## Go with Option 1

In [ ]:
# Bin into clinical age groups
bins = [0, 18, 40, 65, 80, 200]
labels = ["<18", "18-40", "41-65", "66-80", "81+"]

df_age["age_group"] = pd.cut(
    df_age["anchor_age"],
    bins=bins,
    labels=labels,
    right=True
)

# # One-hot encode if your model needs it (tree-based models often prefer ordinal)
# age_dummies = pd.get_dummies(df_age["age_group"], prefix="age_grp")
# df_age = pd.concat([df_age, age_dummies], axis=1)

In [ ]:
# Normalize continuous age
scaler = MinMaxScaler()
df_age["anchor_age_norm"] = scaler.fit_transform(df_age[["anchor_age"]])

In [ ]:
# Final cleaned table
df_age_final = df_age[[
    "subject_id",
    "anchor_age",           # original continuous feature (91 = sentinel for ≥90)
    "anchor_age_norm",      # min-max scaled for RL input
    "age_group",            # categorical label
    *[c for c in df_age.columns if c.startswith("age_grp_")]
]]

print(df_age_final.head())
print(df_age_final.shape)

In [ ]:
from huggingface_hub import login
login('')

In [ ]:
from datasets import Dataset, DatasetInfo

PHYSIONET_LICENSE = "PhysioNet Credentialed Health Data License 1.5.0 — https://physionet.org/content/mimiciv/view-license/3.1/"

age_info = DatasetInfo(
    description=(
        "Patient age features derived from the MIMIC-IV hosp.patients table. "
        "One row per patient (subject_id). "
        "anchor_age is retained as-is, including the MIMIC-IV sentinel value of 91 "
        "which represents patients aged 90 or older (HIPAA de-identification). "
        "Includes min-max normalized age (anchor_age_norm), a clinical age group "
        "categorical (age_group: <18, 18-40, 41-65, 66-80, 81+), and one-hot "
        "encoded age group columns for model input."
    ),
    license=PHYSIONET_LICENSE,
)

ds_age = Dataset.from_pandas(df_age_final, split='age_features', info=age_info)
ds_age.push_to_hub("ADS599-Capstone/modeling_data", config_name="age_features", data_dir="modeling_data")
print("Pushed age_features to HuggingFace Hub.")